# Task 3 — LightGBM point forecast

Two versions of the same model, with the same calendar and lag features and the same hyperparameters.

1. `lgbm_noweather` — calendar and lags, with no weather and without the Terna column.
2. `lgbm_obs_upper_bound` — the same features, plus the six ERA5 temperatures and degree-hours derived from those temperatures at the target hour. This is an upper bound: ERA5 at the target hour was not known at 10:00 on the previous day.

The forecast file is not available. This run does not download it, does not replace it with ERA5, and does not write `lgbm_fc_gfs` predictions.

2025 is not used for training, for choosing hyperparameters, or for the score. Hyperparameters are chosen only on local-day folds inside the training set.


In [ ]:
from pathlib import Path
import sys

import lightgbm as lgb
import numpy as np
import pandas as pd

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "split_config.py").exists():
        sys.path.insert(0, str(candidate))
        REPO = candidate
        break
else:
    raise FileNotFoundError("split_config.py not found from the current directory.")

from split_config import (
    FORECAST_COVERAGE_VERIFIED,
    LOCAL_TZ,
    OBSERVED_TEMP_COLUMNS,
    TARGET_COL,
    TERNA_COL,
    add_lags,
    chronological_local_day_folds,
    load_data,
    load_observed_weather,
    metrics,
    rows_on_local_days,
    seasonal_average_prediction,
    seasonal_naive,
    split,
    validation_index,
    check_predictions,
)

if FORECAST_COVERAGE_VERIFIED:
    raise RuntimeError("Forecast coverage is still closed in this notebook.")

df = load_data()
lagged = add_lags(df)
train, val = split(lagged)
print("train", len(train), "val", len(val))
print("forecast coverage verified:", FORECAST_COVERAGE_VERIFIED)
print("val years", sorted(set(val.index.tz_convert(LOCAL_TZ).year)))


## Features

Local hour and day of week enter as sine and cosine. `is_holiday` stays the CSV flag, including the November Sundays already in the file.

A holiday eve is `is_holiday` on the next local day. A bridge day is a weekday that is not itself a holiday and that sits between two holidays, or between a holiday and the weekend. These are calendar labels, not a second holiday list. The eve of 31 December 2024 uses only the `is_holiday` flag of 1 January, not the 2025 load.

The lags are the UTC shifts of 48, 168 and 336 hours. Degree-hours in the upper-bound version use the mean of the six temperatures at the target hour, with a base of 18 °C. There is no rolling mean.


In [ ]:
DEGREE_BASE_C = 18.0
COMMON_FEATURES = [
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month",
    "is_weekend",
    "is_holiday",
    "is_summer",
    "is_christmas_period",
    "is_holiday_eve",
    "is_bridge",
    "actual_load_mw_lag48h",
    "actual_load_mw_lag168h",
    "actual_load_mw_lag336h",
]
WEATHER_FEATURES = [
    *OBSERVED_TEMP_COLUMNS,
    "heating_degree_c",
    "cooling_degree_c",
]


def add_calendar_indicators(frame: pd.DataFrame) -> pd.DataFrame:
    """Holiday eve and bridge days from the CSV flags. No second holiday list."""
    out = frame.copy()
    local_day = out.index.tz_convert(LOCAL_TZ).floor("D")
    daily = (
        pd.DataFrame(
            {
                "is_holiday": out["is_holiday"].to_numpy(),
                "is_weekend": out["is_weekend"].to_numpy(),
                "dow": out["day_of_week"].to_numpy(),
            },
            index=local_day,
        )
        .groupby(level=0)
        .first()
        .sort_index()
    )
    next_holiday = daily["is_holiday"].shift(-1)
    prev_holiday = daily["is_holiday"].shift(1)
    next_weekend = daily["is_weekend"].shift(-1)
    prev_weekend = daily["is_weekend"].shift(1)
    bridge = (
        daily["dow"].lt(5)
        & daily["is_holiday"].eq(0)
        & (
            (prev_holiday.eq(1) & next_holiday.eq(1))
            | (prev_holiday.eq(1) & next_weekend.eq(1))
            | (prev_weekend.eq(1) & next_holiday.eq(1))
        )
    )
    out["is_holiday_eve"] = next_holiday.fillna(0).astype(int).loc[local_day].to_numpy()
    out["is_bridge"] = bridge.astype(int).loc[local_day].to_numpy()
    out["hour_sin"] = np.sin(2 * np.pi * out["hour_local"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour_local"] / 24)
    out["dow_sin"] = np.sin(2 * np.pi * out["day_of_week"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["day_of_week"] / 7)
    return out


def observed_temperature_features(index, weather: pd.DataFrame) -> pd.DataFrame:
    """Same-hour ERA5 temperatures and degree-hours. Upper bound only."""
    temps = weather.reindex(index)[list(OBSERVED_TEMP_COLUMNS)]
    if temps.isna().any().any():
        raise ValueError("ERA5 temperatures do not cover every target hour.")
    mean_temp = temps.mean(axis=1)
    temps = temps.copy()
    temps["heating_degree_c"] = (DEGREE_BASE_C - mean_temp).clip(lower=0)
    temps["cooling_degree_c"] = (mean_temp - DEGREE_BASE_C).clip(lower=0)
    return temps


lagged = add_calendar_indicators(lagged)
train, val = split(lagged)
weather = load_observed_weather()
train_weather = observed_temperature_features(train.index, weather)
val_weather = observed_temperature_features(val.index, weather)

used = pd.concat([train, val])
if not np.isfinite(used[COMMON_FEATURES].to_numpy(dtype=float)).all():
    raise ValueError("A calendar or lag feature is missing on train or validation.")
if train.index.tz_convert(LOCAL_TZ).year.max() > 2023:
    raise ValueError("Training rows go past local year 2023.")
if set(val.index.tz_convert(LOCAL_TZ).year) != {2024}:
    raise ValueError("Validation is not local year 2024.")

print("common features", len(COMMON_FEATURES))
print("upper-bound features", len(COMMON_FEATURES) + len(WEATHER_FEATURES))
print("holiday eves in validation", int(val["is_holiday_eve"].sum()))
print("bridge hours in validation", int(val["is_bridge"].sum()))


## Hyperparameters

One grid is scored on the no-weather features. Each candidate is evaluated on the four expanding local-day folds of the training set. The criterion is the MAE pooled over all the hours in those folds (not the mean of per-fold MAEs). 2024 is not used for the choice. The chosen hyperparameters are then used for the ERA5 version as well.

The grid is wider than a first pass would suggest: the best candidate of a narrower grid sat on its edge for `n_estimators` and `learning_rate`, so the range now extends beyond those values, and `num_leaves` goes below 31. The notebook prints a warning if the chosen value is still on the edge of the grid.

In [ ]:
folds = chronological_local_day_folds(train.index, n_splits=4)
print("folds", [(len(train_days), len(fold_days)) for train_days, fold_days in folds])

GRID_VALUES = {
    "n_estimators": (200, 400, 800),
    "learning_rate": (0.05, 0.1, 0.2),
    "num_leaves": (15, 31, 63),
}
INT_PARAMS = ("n_estimators", "num_leaves")

grid = [
    {"n_estimators": n_estimators, "learning_rate": learning_rate, "num_leaves": num_leaves}
    for n_estimators in GRID_VALUES["n_estimators"]
    for learning_rate in GRID_VALUES["learning_rate"]
    for num_leaves in GRID_VALUES["num_leaves"]
]


def fold_mae(params, features):
    """MAE pooled over all the hours of the validation folds."""
    abs_errors = []
    for train_days, fold_days in folds:
        inner_train = rows_on_local_days(train, train_days)
        inner_val = rows_on_local_days(train, fold_days)
        model = lgb.LGBMRegressor(
            **params,
            objective="regression",
            random_state=0,
            verbosity=-1,
            n_jobs=-1,
        )
        model.fit(inner_train[features], inner_train[TARGET_COL])
        predicted = model.predict(inner_val[features])
        abs_errors.append(np.abs(inner_val[TARGET_COL].to_numpy() - predicted))
    return float(np.concatenate(abs_errors).mean())


search_rows = []
for params in grid:
    score = fold_mae(params, COMMON_FEATURES)
    search_rows.append({**params, "mae": score})
    print(params, "MAE", round(score, 3))

search = pd.DataFrame(search_rows).sort_values("mae").reset_index(drop=True)
best = {
    name: (int(search.loc[0, name]) if name in INT_PARAMS else float(search.loc[0, name]))
    for name in GRID_VALUES
}
print("chosen", best)

for name, values in GRID_VALUES.items():
    if best[name] in (min(values), max(values)):
        print(f"warning: chosen {name}={best[name]} is on the edge of the grid {values}")

## Fit and 2024 validation forecast

Each version is refit on the whole training set with the hyperparameters already fixed, and predicts every hour of 2024.


In [ ]:
def fit_model(x_train, y_train):
    model = lgb.LGBMRegressor(
        **best,
        objective="regression",
        random_state=0,
        verbosity=-1,
        n_jobs=-1,
    )
    model.fit(x_train, y_train)
    return model


model_noweather = fit_model(train[COMMON_FEATURES], train[TARGET_COL])
pred_noweather = model_noweather.predict(val[COMMON_FEATURES])

x_train_obs = pd.concat([train[COMMON_FEATURES], train_weather[WEATHER_FEATURES]], axis=1)
x_val_obs = pd.concat([val[COMMON_FEATURES], val_weather[WEATHER_FEATURES]], axis=1)
model_obs = fit_model(x_train_obs, train[TARGET_COL])
pred_obs = model_obs.predict(x_val_obs)

if len(pred_noweather) != len(val) or len(pred_obs) != len(val):
    raise ValueError("A model did not score every validation hour.")
print("validation hours", len(val))


## CSV files and comparison

The three references are the same-local-hour naive forecast, the hour-of-week mean estimated on the training set, and Terna's forecast. Terna is not a feature. All five models are scored on the same hours, the ones where the seasonal naive exists, so that `n` is identical in every row. The CSV files still hold every hour of 2024.


In [ ]:
OUT_DIR = REPO / "notebooks" / "martina" / "predictions"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def prediction_frame(y_pred, model_name) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "timestamp": val.index.tz_convert("UTC").strftime("%Y-%m-%dT%H:%M:%SZ"),
            "y_true": val[TARGET_COL].to_numpy(),
            "y_pred": y_pred,
            "model": model_name,
        }
    )


paths = {
    "lgbm_noweather": OUT_DIR / "lgbm_noweather.csv",
    "lgbm_obs_upper_bound": OUT_DIR / "lgbm_obs_upper_bound.csv",
}
prediction_frame(pred_noweather, "lgbm_noweather").to_csv(paths["lgbm_noweather"], index=False)
prediction_frame(pred_obs, "lgbm_obs_upper_bound").to_csv(
    paths["lgbm_obs_upper_bound"], index=False
)

expected = validation_index(df)
for name, path in paths.items():
    checked = check_predictions(path, expected, reference_y=df[TARGET_COL])
    print(name, "checked rows", len(checked))

history = df.loc[: val.index.max()]
naive = seasonal_naive(history).reindex(val.index)
seasonal_avg = pd.Series(np.asarray(seasonal_average_prediction(train, val), dtype=float), index=val.index)
terna = val[TERNA_COL]

# All the models are scored on the same hours: those where the seasonal naive exists.
common = val.index[naive.notna()]
print("hours in the comparison", len(common), "of", len(val))
y_common = val.loc[common, TARGET_COL]

candidates = {
    "lgbm_noweather": pd.Series(pred_noweather, index=val.index),
    "lgbm_obs_upper_bound": pd.Series(pred_obs, index=val.index),
    "seasonal_naive": naive,
    "seasonal_avg": seasonal_avg,
    "terna": terna,
}
rows = [
    {"model": name, **metrics(y_common, prediction.loc[common])}
    for name, prediction in candidates.items()
]
comparison = pd.DataFrame(rows)[["model", "n", "MAE", "RMSE"]]
print(comparison.to_string(index=False, float_format=lambda value: f"{value:,.2f}"))

## Feature importance

Gain importance of the no-weather model, the operational version in this notebook. The share is the fraction of total gain.

Read this together with the comparison table: the weekly lag `actual_load_mw_lag168h` is expected to carry most of the gain (about 80% in the previous run), so the model behaves largely as a weekly seasonal naive corrected by calendar features. Terna's forecast remains well ahead of this no-weather model, because it uses information that is not in these features. The gap to Terna is the main limit of this version.


In [ ]:
gain = pd.Series(
    model_noweather.booster_.feature_importance(importance_type="gain"),
    index=COMMON_FEATURES,
    name="gain_share",
).sort_values(ascending=False)
gain = gain / gain.sum()
print(gain.to_string(float_format=lambda value: f"{value:.3f}"))
